# Lab L5.1 · Deploy the MIZAN fraud service — feature module, MLflow, model card, FastAPI and a CI gate
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 5 · ⏱ 90 min · **Level 1 · From scratch**

Package the Day 3 fraud model as a production-grade service: one feature module for training and serving, tracked experiments, a registered champion, a model card, a validated API, a load test and a release gate.

Follow **L5.1_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Move feature code into a module and prove parity

1. Create the package folder `checkpoints/mizan_fraud/` (+ `__init__.py`, `model/`) and write `features.py` containing `build_fraud_features`, `NUMF`, `CATF` with a `%%writefile` cell.
2. Import it; rebuild features from the raw file; compare with `card_features.parquet` on 50k random txn_ids (`np.allclose`, NaN-aware).
3. Now rebuild using **only September** raw data and measure how many `card_txn_index` / `secs_since_prev` values differ. Explain why.

In [ ]:
# Your code for step 1


## Step 2 · Track candidate models in MLflow

1. Point MLflow at `sqlite:///checkpoints/mlflow/mlflow.db`; create/get experiment `mizan-fraud` with artefacts under `checkpoints/mlflow/artifacts`.
2. Undersample train (all fraud + 250k legit). For 3 configs of (`n_estimators`, `min_samples_leaf`, `max_features`) fit an RF pipeline, compute `at_budget` on valid, and log params + metrics + training time in a run.
3. Query the runs into a DataFrame with `mlflow.search_runs` and pick the best by `PR_AUC`.

In [ ]:
# Your code for step 2


## Step 3 · Refit, set the alert threshold, write the model card, register the champion

1. Compute `threshold` = the 99.5th percentile of the candidate’s validation scores.
2. Refit `best_cfg` on undersampled train+valid; evaluate `at_budget` on test.
3. Save `fraud_rf.joblib` and `model_card.json` into `mizan_fraud/model/`.
4. Log a final run with the artefacts, register it as `mizan-fraud`, and set alias `champion`.

In [ ]:
# Your code for step 3


## Step 4 · Serve it: FastAPI with Pydantic validation

1. Write `mizan_fraud/app.py` with `%%writefile`: Pydantic `Txn` schema, endpoints `/health`, `/score`, `/score_batch`.
2. With `TestClient`: call `/health`; score 3 real test transactions (one known fraud); send 2 invalid payloads (negative amount, unknown channel) and confirm HTTP 422.
3. Note the command to run it live: `uvicorn mizan_fraud.app:app --port 8000` from the `checkpoints` folder.

In [ ]:
# Your code for step 4


## Step 5 · Load test: latency and throughput

1. Send 300 single `/score` requests; record wall-clock latency; report p50/p95/p99.
2. Send one `/score_batch` request with 5,000 transactions; report transactions/second.

In [ ]:
# Your code for step 5


## Step 6 · Nightly batch job on new raw data + CI release gate

1. Batch job: concatenate raw history (Aug–Sep) with the first production week (1–7 Oct), build features with the module, keep October rows, score them, count alerts per day.
2. Write `release_gate(model, card)` checking: test PR-AUC ≥ 0.85, test alert rate between 0.3% and 0.8%, p95 latency < 150 ms, no ID/PII features, threshold in (0,1). Save `release_gate.json` and print DEPLOY/BLOCK.

In [ ]:
# Your code for step 6
